In [ ]:
import os
import re
import pandas as pd
import win32com.client
from datetime import datetime, timedelta

In [12]:

ruta_descarga = (
    r"\\192.168.2.12\SQLServer Compartido"
    r"\001_gestiones_py_vigente\Adjuntos_Outlook"
)

os.makedirs(ruta_descarga, exist_ok=True)

# Archivo donde se guardará el control
ruta_control = os.path.join(
    ruta_descarga,
    "control_descarga.csv"
)

remitentes_permitidos = {
    "bmontenegroq@efectiva.com.pe",
    # "analista.datos@targetoutsourcing.com.pe"
}


# fecha_inicio = datetime.now().replace(
#     hour=0,
#     minute=0,
#     second=0,
#     microsecond=0
# )


fecha_inicio = (
    datetime.now()
    .replace(hour=0, minute=0, second=0, microsecond=0)
    - timedelta(days=1)
)

fecha_fin = fecha_inicio + timedelta(days=1)
print(fecha_inicio)
print(fecha_fin)

extensiones_permitidas = (
    ".xlsx",
    ".xls",
    ".csv",
    ".txt",
    ".zip"
)


def obtener_correo_remitente(mensaje):
    """
    Obtiene el correo SMTP del remitente.
    También funciona con cuentas corporativas Exchange.
    """
    try:
        if mensaje.SenderEmailType == "EX":

            usuario_exchange = mensaje.Sender.GetExchangeUser()

            if usuario_exchange:
                return usuario_exchange.PrimarySmtpAddress

        return mensaje.SenderEmailAddress

    except Exception:
        return mensaje.SenderEmailAddress


def limpiar_nombre_archivo(nombre):
    """
    Reemplaza caracteres no permitidos en Windows.
    """
    return re.sub(r'[<>:"/\\|?*]', "_", nombre)


def obtener_ruta_unica(carpeta, nombre_archivo):
    """
    Evita reemplazar archivos ya existentes.
    """
    nombre_archivo = limpiar_nombre_archivo(nombre_archivo)

    # nombre, extension = os.path.splitext(nombre_archivo)
    ruta_final = os.path.join(carpeta, nombre_archivo)

    # contador = 1

    # while os.path.exists(ruta_final):

    #     nuevo_nombre = f"{nombre}_{contador}{extension}"

    #     ruta_final = os.path.join(
    #         carpeta,
    #         nuevo_nombre
    #     )

    #     contador += 1

    return ruta_final


columnas_control = [
    "correo_remitente",
    "asunto",
    "fecha_correo",
    "nombre_archivo",
    "ruta_archivo"
]


2026-07-27 00:00:00
2026-07-28 00:00:00


In [13]:
# leer archivo control

if os.path.exists(ruta_control):

    df_control = pd.read_csv(
        ruta_control,
        sep=";",
        encoding="utf-8-sig"
    )

    df_control["fecha_correo"] = pd.to_datetime(
        df_control["fecha_correo"],
        errors="coerce"
    )

    df_control = df_control.dropna(
        subset=["fecha_correo"]
    ).copy()

    df_control["correo_remitente"] = (
        df_control["correo_remitente"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df_control["asunto"] = (
        df_control["asunto"]
        .astype(str)
        .str.strip()
    )

    print(
        f"Control encontrado: {len(df_control)} registros"
    )
else:

    df_control = pd.DataFrame(
        columns=columnas_control
    )

    print(
        "El archivo de control no existe. "
        "Se creará al finalizar."
    )

if not df_control.empty:

    df_control["fecha_actualizacion"] = (
        df_control["fecha_correo"].dt.date
    )

else:

    df_control["fecha_actualizacion"] = pd.Series(
        dtype="object"
    )
    
df_control.head()

El archivo de control no existe. Se creará al finalizar.


,correo_remitente,asunto,fecha_correo,nombre_archivo,ruta_archivo,fecha_actualizacion


In [14]:
# CONECTAR CON OUTLOOK
outlook = win32com.client.Dispatch(
    "Outlook.Application"
)

namespace = outlook.GetNamespace("MAPI")
bandeja_entrada = namespace.GetDefaultFolder(6)
mensajes = bandeja_entrada.Items

filtro = (
    "[ReceivedTime] >= '" + fecha_inicio.strftime("%m/%d/%Y %I:%M %p") + "' "
    "AND "
    "[ReceivedTime] < '" + fecha_fin.strftime("%m/%d/%Y %I:%M %p") + "'"
)
mensajes = mensajes.Restrict(filtro)


mensajes.Sort("[ReceivedTime]", True)
print("Correos encontrados:", mensajes.Count)

Correos encontrados: 41


In [19]:
df_control.head()

,correo_remitente,asunto,fecha_correo,nombre_archivo,ruta_archivo,fecha_actualizacion


In [23]:
# RECORRER Y DESCARGAR

registro_descargas = []

correos_revisados_por_dia = set()

for mensaje in mensajes:

    try:
        # 43 = MailItem
        if mensaje.Class != 43:
            continue
        fecha_correo = mensaje.ReceivedTime.replace(tzinfo=None)
        correo_remitente = obtener_correo_remitente(mensaje)
            
        if not correo_remitente:
            continue
        correo_remitente = correo_remitente.strip().lower()
            
        if correo_remitente not in remitentes_permitidos:
            continue

        asunto = (mensaje.Subject or "Sin asunto").strip()

        # fecha_dia = fecha_correo.date()

        existe = (
            (df_control['fecha_correo'] == fecha_correo) &
            (df_control['nombre_archivo'] == nombre_archivo)
        ).any()

        if not existe:
            continue
        # Identificador por remitente, asunto y día
        clave_dia = (
            correo_remitente,
            asunto.lower()
        )

        if clave_dia in correos_revisados_por_dia:
            continue

        correos_revisados_por_dia.add(clave_dia)


        # VALIDAR QUE TENGA ADJUNTOS

        if mensaje.Attachments.Count == 0:
            continue

        # # ====================================================
        # # ====================================================
        # # ====================================================

        for posicion in range(
            1,
            mensaje.Attachments.Count + 1
        ):

            adjunto = mensaje.Attachments.Item(
                posicion
            )

            nombre_archivo = adjunto.FileName

            if not nombre_archivo.lower().endswith(extensiones_permitidas):
                continue

            ruta_final = obtener_ruta_unica(
                ruta_descarga,
                nombre_archivo
            )

            adjunto.SaveAsFile(ruta_final)


            registro_descargas.append({
                "correo_remitente": correo_remitente,
                "asunto": asunto,
                "fecha_correo": fecha_correo,
                "nombre_archivo": nombre_archivo,
                "ruta_archivo": ruta_final,
                "fecha_actualizacion":fecha_inicio,
            })

            print(
                f"Descargado: {nombre_archivo}"
            )


    except Exception as error:
        
        print(
            "Error procesando correo:",
            error
        )

if not registro_descargas:
    print("No se encontró ningún correo")


No se encontró ningún correo


In [ ]:
registro_descargas

In [ ]:

        if not df_control.empty:

            existe_en_control = (
                (df_control["correo_remitente"]== correo_remitente)&
                (df_control["asunto"].str.lower()== asunto.lower())&
                (df_control["fecha_correo"]== fecha_correo)
            ).any()
        else:
            existe_en_control = False

In [147]:
df_control.head()

,correo_remitente,asunto,fecha_correo,nombre_archivo,ruta_archivo,fecha_actualizacion


In [ ]:

        # # ====================================================
        # # ACTUALIZAR CONTROL SOLO SI DESCARGÓ ALGO
        # # ====================================================

        # if cantidad_descargada > 0:

        #     # Quitar el registro anterior del mismo
        #     # remitente + asunto + día
        #     if not df_control.empty:

        #         condicion_mismo_dia = (
        #             (
        #                 df_control["correo_remitente"]
        #                 == correo_remitente
        #             )
        #             &
        #             (
        #                 df_control["asunto"]
        #                 .str.lower()
        #                 == asunto.lower()
        #             )
        #             &
        #             (
        #                 df_control["fecha_dia"]
        #                 == fecha_dia
        #             )
        #         )

        #         df_control = df_control[
        #             ~condicion_mismo_dia
        #         ].copy()

        #     nuevo_control = pd.DataFrame([{
        #         "correo_remitente": correo_remitente,
        #         "asunto": asunto,
        #         "fecha_correo": fecha_correo,
        #         "fecha_dia": fecha_dia
        #     }])

        #     df_control = pd.concat(
        #         [
        #             df_control,
        #             nuevo_control
        #         ],
        #         ignore_index=True
        #     )


In [117]:
registro_descargas


[{'fecha_correo': pywintypes.datetime(2026, 7, 27, 11, 10, 23),
  'nombre_remitente': 'Bryan Victor Montenegro Quiroz',
  'correo_remitente': 'bmontenegroq@efectiva.com.pe',
  'asunto': 'Reporte de Canal Julio 2026 - TARGET',
  'nombre_archivo': 'TARGET_EFECTIVO_202607.xlsx',
  'ruta_archivo': '\\\\192.168.2.12\\SQLServer Compartido\\001_gestiones_py_vigente\\Adjuntos_Outlook\\TARGET_EFECTIVO_202607.xlsx'},
 {'fecha_correo': pywintypes.datetime(2026, 7, 27, 11, 10, 13),
  'nombre_remitente': 'Bryan Victor Montenegro Quiroz',
  'correo_remitente': 'bmontenegroq@efectiva.com.pe',
  'asunto': 'Reporte de Canal Julio 2026 - TARGET EFECTINEGOCIO',
  'nombre_archivo': 'Target_Efectinegocio_202607.xlsx',
  'ruta_archivo': '\\\\192.168.2.12\\SQLServer Compartido\\001_gestiones_py_vigente\\Adjuntos_Outlook\\Target_Efectinegocio_202607.xlsx'}]

In [48]:

if not df_control.empty:

    # Ordenar desde la fecha más reciente
    df_control = df_control.sort_values(
        by="fecha_correo",
        ascending=False
    )

    # Conservar solo las columnas solicitadas
    df_control_exportar = df_control[
        [
            "correo_remitente",
            "asunto",
            "fecha_correo"
        ]
    ].copy()

    # Formato: 27/07/2026 11:10:23
    df_control_exportar["fecha_correo"] = (
        df_control_exportar["fecha_correo"]
        .dt.strftime("%d/%m/%Y %H:%M:%S")
    )

    df_control_exportar.to_csv(
        ruta_control,
        index=False,
        sep=";",
        encoding="utf-8-sig"
    )

    print("=" * 90)
    print("Control actualizado:")
    print(ruta_control)


# ============================================================
# 7. RESULTADO DE ESTA EJECUCIÓN
# ============================================================

df_descargas = pd.DataFrame(
    registro_descargas
)

if not df_descargas.empty:

    display(df_descargas)

    print(
        f"\nTotal descargado en esta ejecución: "
        f"{len(df_descargas)}"
    )

else:

    print(
        "\nNo hubo nuevos archivos para descargar."
    )

Control actualizado:
\\192.168.2.12\SQLServer Compartido\001_gestiones_py_vigente\Adjuntos_Outlook\control_descarga.csv


,fecha_correo,nombre_remitente,correo_remitente,asunto,nombre_archivo,ruta_archivo
0,2026-07-27 11:10:23,Bryan Victor Montenegro Quiroz,bmontenegroq@efectiva.com.pe,Reporte de Canal Julio 2026 - TARGET,TARGET_EFECTIVO_202607.xlsx,\\192.168.2.12\SQLServer Compartido\001_gestiones_py_vigente\Adjuntos_Outlook\Target_Efectinegocio_202607_2.xlsx
1,2026-07-27 11:10:13,Bryan Victor Montenegro Quiroz,bmontenegroq@efectiva.com.pe,Reporte de Canal Julio 2026 - TARGET EFECTINEGOCIO,Target_Efectinegocio_202607.xlsx,\\192.168.2.12\SQLServer Compartido\001_gestiones_py_vigente\Adjuntos_Outlook\Target_Efectinegocio_202607_2.xlsx



Total descargado en esta ejecución: 2


In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)


In [24]:
import os
import re
import pandas as pd
import win32com.client
from datetime import datetime
ruta_descarga = (
    r"\\192.168.2.12\SQLServer Compartido"
    r"\001_gestiones_py_vigente\Adjuntos_Outlook"
)

os.makedirs(ruta_descarga, exist_ok=True)

remitentes_permitidos = {
    "bmontenegroq@efectiva.com.pe",
    "analista.datos@targetoutsourcing.com.pe"
}

fecha_inicio = datetime(2026, 7, 27)

# Tipos de archivo que deseas descargar
extensiones_permitidas = (
    ".xlsx",
    ".xls",
    ".csv",
    ".txt",
    ".zip"
)



def obtener_correo_remitente(mensaje):
    try:
        if mensaje.SenderEmailType == "EX":
            usuario_exchange = mensaje.Sender.GetExchangeUser()

            if usuario_exchange:
                return usuario_exchange.PrimarySmtpAddress

        return mensaje.SenderEmailAddress

    except Exception:
        return mensaje.SenderEmailAddress


def limpiar_nombre_archivo(nombre):
    return re.sub(r'[<>:"/\\|?*]', "_", nombre)


def obtener_ruta_unica(carpeta, nombre_archivo):
    """
    Evita reemplazar archivos que ya existen.
    """
    nombre_archivo = limpiar_nombre_archivo(nombre_archivo)

    nombre, extension = os.path.splitext(nombre_archivo)
    ruta_final = os.path.join(carpeta, nombre_archivo)

    contador = 1

    while os.path.exists(ruta_final):
        nuevo_nombre = f"{nombre}_{contador}{extension}"
        ruta_final = os.path.join(carpeta, nuevo_nombre)
        contador += 1

    return ruta_final



In [ ]:

# ============================================================
# 3. CONECTAR CON OUTLOOK
# ============================================================

outlook = win32com.client.Dispatch("Outlook.Application")
namespace = outlook.GetNamespace("MAPI")

# 6 = Bandeja de entrada
bandeja_entrada = namespace.GetDefaultFolder(6)

mensajes = bandeja_entrada.Items
mensajes.Sort("[ReceivedTime]", True)


# ============================================================
# 4. DESCARGAR ADJUNTOS
# ============================================================

registro_descargas = []

for mensaje in mensajes:

    try:
        # 43 = correo electrónico
        if mensaje.Class != 43:
            continue

        fecha_correo = mensaje.ReceivedTime.replace(tzinfo=None)

        # Como está ordenado desde el más reciente,
        # al llegar a fechas anteriores se termina.
        if fecha_correo < fecha_inicio:
            break

        correo_remitente = obtener_correo_remitente(mensaje)

        if not correo_remitente:
            continue

        correo_remitente = correo_remitente.strip().lower()

        # Filtrar únicamente los dos remitentes
        if correo_remitente not in remitentes_permitidos:
            continue

        asunto = mensaje.Subject or "Sin asunto"

        if mensaje.Attachments.Count == 0:
            continue

        print("=" * 90)
        print("Fecha:", fecha_correo)
        print("Remitente:", mensaje.SenderName)
        print("Correo:", correo_remitente)
        print("Asunto:", asunto)
        print("Adjuntos:", mensaje.Attachments.Count)

        for posicion in range(1, mensaje.Attachments.Count + 1):

            adjunto = mensaje.Attachments.Item(posicion)
            nombre_archivo = adjunto.FileName

            # Filtrar por extensión
            if not nombre_archivo.lower().endswith(
                extensiones_permitidas
            ):
                print(f"Omitido: {nombre_archivo}")
                continue

            ruta_final = obtener_ruta_unica(
                ruta_descarga,
                nombre_archivo
            )

            adjunto.SaveAsFile(ruta_final)

            registro_descargas.append({
                "fecha_correo": fecha_correo,
                "nombre_remitente": mensaje.SenderName,
                "correo_remitente": correo_remitente,
                "asunto": asunto,
                "nombre_archivo": nombre_archivo,
                "ruta_archivo": ruta_final
            })

            print(f"Descargado: {nombre_archivo}")

    except Exception as error:
        print("Error procesando correo:", error)


# ============================================================
# 5. RESULTADO EN PANDAS
# ============================================================

df_descargas = pd.DataFrame(registro_descargas)

if not df_descargas.empty:
    display(df_descargas)
    print(f"\nTotal de archivos descargados: {len(df_descargas)}")
else:
    print("No se encontraron adjuntos para esos remitentes.")

Fecha: 2026-07-27 11:10:23
Remitente: Bryan Victor Montenegro Quiroz
Correo: bmontenegroq@efectiva.com.pe
Asunto: Reporte de Canal Julio 2026 - TARGET
Adjuntos: 2
Omitido: image003.jpg
Descargado: TARGET_EFECTIVO_202607.xlsx
Fecha: 2026-07-27 11:10:13
Remitente: Bryan Victor Montenegro Quiroz
Correo: bmontenegroq@efectiva.com.pe
Asunto: Reporte de Canal Julio 2026 - TARGET EFECTINEGOCIO
Adjuntos: 2
Omitido: image003.jpg
Descargado: Target_Efectinegocio_202607.xlsx


,fecha_correo,nombre_remitente,correo_remitente,asunto,nombre_archivo,ruta_archivo
0,2026-07-27 11:10:23,Bryan Victor Montenegro Quiroz,bmontenegroq@efectiva.com.pe,Reporte de Canal Julio 2026 - TARGET,TARGET_EFECTIVO_202607.xlsx,\\192.168.2.12\SQLServer Compartido\001_gestiones_py_vigente\Adjuntos_Outlook\TARGET_EFECTIVO_202607.xlsx
1,2026-07-27 11:10:13,Bryan Victor Montenegro Quiroz,bmontenegroq@efectiva.com.pe,Reporte de Canal Julio 2026 - TARGET EFECTINEGOCIO,Target_Efectinegocio_202607.xlsx,\\192.168.2.12\SQLServer Compartido\001_gestiones_py_vigente\Adjuntos_Outlook\Target_Efectinegocio_202607.xlsx



Total de archivos descargados: 2


In [2]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from unidecode import unidecode
from sqlalchemy import create_engine
from sqlalchemy import text

fecha_mes_base='2026-07-01'

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

tipi_cond1='RECLUTAMIENTO'
servidor_01=64

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

server_sql = server_zeus
db_sql = "ODIN"
user_sql = user_zeus
pwd_sql = pwd_zeus
engine_zeus = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

In [21]:
import os
import re
import win32com.client

ruta_descarga='\\192.168.2.12\\SQLServer Compartido\\001_gestiones_py_vigente\\Adjuntos_Outlook'
os.makedirs(ruta_descarga, exist_ok=True)

outlook = win32com.client.Dispatch("Outlook.Application")
namespace = outlook.GetNamespace("MAPI")

# 6 = Bandeja de entrada
bandeja_entrada = namespace.GetDefaultFolder(6)
mensajes = bandeja_entrada.Items

# Ordenar desde el correo más reciente
mensajes.Sort("[ReceivedTime]", True)


In [22]:

def limpiar_nombre_archivo(nombre):
    """
    Elimina caracteres no permitidos en nombres de archivos de Windows.
    """
    return re.sub(r'[<>:"/\\|?*]', "_", nombre)



def obtener_ruta_unica(carpeta, nombre_archivo):
    """
    Si el archivo ya existe, agrega _1, _2, etc.
    """
    nombre_archivo = limpiar_nombre_archivo(nombre_archivo)

    nombre, extension = os.path.splitext(nombre_archivo)
    ruta_final = os.path.join(carpeta, nombre_archivo)

    contador = 1

    while os.path.exists(ruta_final):
        nuevo_nombre = f"{nombre}_{contador}{extension}"
        ruta_final = os.path.join(carpeta, nuevo_nombre)
        contador += 1

    return ruta_final



In [ ]:

# ============================================================
# DESCARGAR ADJUNTOS
# ============================================================

archivos_descargados = []

for mensaje in mensajes:

    try:
        if mensaje.Class != 43:
            continue

        if mensaje.Attachments.Count == 0:
            continue

        asunto = mensaje.Subject or "Sin asunto"
        remitente = mensaje.SenderEmailAddress or "Sin remitente"
        fecha_correo = mensaje.ReceivedTime

        for posicion in range(1, mensaje.Attachments.Count + 1):

            adjunto = mensaje.Attachments.Item(posicion)
            nombre_archivo = adjunto.FileName

            ruta_final = obtener_ruta_unica(
                ruta_descarga,
                nombre_archivo
            )

            adjunto.SaveAsFile(ruta_final)

            archivos_descargados.append({
                "fecha_correo": fecha_correo,
                "remitente": remitente,
                "asunto": asunto,
                "archivo": nombre_archivo,
                "ruta": ruta_final
            })

            print(f"Descargado: {ruta_final}")

    except Exception as error:
        print(f"Error procesando correo: {error}")


print(f"\nTotal descargados: {len(archivos_descargados)}")

In [ ]:
import os
import win32com.client

ruta_descarga = (
    r"\\192.168.2.12\SQLServer Compartido"
    r"\001_gestiones_py_vigente\Adjuntos_Outlook"
)

os.makedirs(ruta_descarga, exist_ok=True)

outlook = win32com.client.Dispatch("Outlook.Application")
namespace = outlook.GetNamespace("MAPI")
Bryan Victor Montenegro Quiroz <bmontenegroq@efectiva.com.pe>
# 6 = Bandeja de entrada
bandeja_entrada = namespace.GetDefaultFolder(6)

mensajes = bandeja_entrada.Items
mensajes.Sort("[ReceivedTime]", True)


# Mostrar los últimos 10 mensajes
contador = 0

for mensaje in mensajes:

    try:
        # 43 significa que el elemento es un correo
        if mensaje.Class != 43:
            continue

        print("=" * 100)
        print("Fecha:", mensaje.ReceivedTime)
        print("Remitente:", mensaje.SenderName)
        print("Correo:", mensaje.SenderEmailAddress)
        print("Asunto:", mensaje.Subject)
        print("Adjuntos:", mensaje.Attachments.Count)
        print("-" * 100)
        print("Contenido:")
        print(mensaje.Body)
        print("=" * 100)

        contador += 1

        if contador >= 10:
            break

    except Exception as error:
        print("Error al leer el mensaje:", error)

Fecha: 2026-07-27 12:01:47+00:00
Remitente: bifinanciera_saas@efectiva.com.pe
Correo: bifinanciera_saas@efectiva.com.pe
Asunto: ERROR: Carga de Gestiones CALL TARJET [23], Corte 27/07/2026 - 2026-07-27 12:01:43
Adjuntos: 0
----------------------------------------------------------------------------------------------------
Contenido:

        
        Fecha de corte 27/07/2026
        ------------------------------------------------------------	
        Procesamiento de interface fallido
        0
        ------------------------------------------------------------
        Se procesaron el 27/07/2026 a horas 12:01:32
        Duracion : 00:00:10
        ------------------------------------------------------------
        Filas nuevas : 0
        Filas sin Procesar : 0
        ------------------------------------------------------------
        Resultado : Error
        Nombre : 85= REVISAR ESTRUCTURA DE INTERFACE /Buffer_Historico_BI/CanalesGestion/Procesada/CG_TARGET_20260727_120132.txt

In [20]:
contador = 0

for mensaje in mensajes:

    try:
        # 43 significa que el elemento es un correo
        if mensaje.Class != 43:
            continue

        print("=" * 100)
        print("Fecha:", mensaje.ReceivedTime)
        print("Remitente:", mensaje.SenderName)
        print("Correo:", mensaje.SenderEmailAddress)

        contador += 1

        if contador >= 10:
            break

    except Exception as error:
        print("Error al leer el mensaje:", error)

Fecha: 2026-07-27 12:01:47+00:00
Remitente: bifinanciera_saas@efectiva.com.pe
Correo: bifinanciera_saas@efectiva.com.pe
Fecha: 2026-07-27 11:38:59+00:00
Remitente: inteligencianegocios@cencosudscotiabank.pe
Correo: inteligencianegocios@cencosudscotiabank.pe
Fecha: 2026-07-27 11:10:23+00:00
Remitente: Bryan Victor Montenegro Quiroz
Correo: bmontenegroq@efectiva.com.pe
Fecha: 2026-07-27 11:10:13+00:00
Remitente: Bryan Victor Montenegro Quiroz
Correo: bmontenegroq@efectiva.com.pe
Fecha: 2026-07-27 11:01:33+00:00
Remitente: bifinanciera_saas@efectiva.com.pe
Correo: bifinanciera_saas@efectiva.com.pe
Fecha: 2026-07-27 11:01:23+00:00
Remitente: reportes.target
Correo: analista1@targetoutsourcing.com.pe
Fecha: 2026-07-27 11:00:32+00:00
Remitente: reportes.target
Correo: analista1@targetoutsourcing.com.pe
Fecha: 2026-07-27 10:52:04+00:00
Remitente: supervisor.efectiva.consumo@targetoutsourcing.com.pe
Correo: supervisor.efectiva.consumo@targetoutsourcing.com.pe
Fecha: 2026-07-27 10:43:07+00:00
R

In [ ]:



mensajes = bandeja_entrada.Items

# Ordenar desde el correo más reciente
mensajes.Sort("[ReceivedTime]", True)


# ============================================================
# FUNCIÓN PARA LIMPIAR EL NOMBRE DEL ARCHIVO
# ============================================================

def limpiar_nombre_archivo(nombre):
    """
    Elimina caracteres no permitidos en nombres de archivos de Windows.
    """
    return re.sub(r'[<>:"/\\|?*]', "_", nombre)


# ============================================================
# FUNCIÓN PARA EVITAR SOBREESCRIBIR ARCHIVOS
# ============================================================

def obtener_ruta_unica(carpeta, nombre_archivo):
    """
    Si el archivo ya existe, agrega _1, _2, etc.
    """
    nombre_archivo = limpiar_nombre_archivo(nombre_archivo)

    nombre, extension = os.path.splitext(nombre_archivo)
    ruta_final = os.path.join(carpeta, nombre_archivo)

    contador = 1

    while os.path.exists(ruta_final):
        nuevo_nombre = f"{nombre}_{contador}{extension}"
        ruta_final = os.path.join(carpeta, nuevo_nombre)
        contador += 1

    return ruta_final


# ============================================================
# DESCARGAR ADJUNTOS
# ============================================================

archivos_descargados = []

for mensaje in mensajes:

    try:
        # Verificar que sea un correo electrónico
        if mensaje.Class != 43:
            continue

        if mensaje.Attachments.Count == 0:
            continue

        asunto = mensaje.Subject or "Sin asunto"
        remitente = mensaje.SenderEmailAddress or "Sin remitente"
        fecha_correo = mensaje.ReceivedTime

        for posicion in range(1, mensaje.Attachments.Count + 1):

            adjunto = mensaje.Attachments.Item(posicion)
            nombre_archivo = adjunto.FileName

            ruta_final = obtener_ruta_unica(
                ruta_descarga,
                nombre_archivo
            )

            adjunto.SaveAsFile(ruta_final)

            archivos_descargados.append({
                "fecha_correo": fecha_correo,
                "remitente": remitente,
                "asunto": asunto,
                "archivo": nombre_archivo,
                "ruta": ruta_final
            })

            print(f"Descargado: {ruta_final}")

    except Exception as error:
        print(f"Error procesando correo: {error}")


print(f"\nTotal descargados: {len(archivos_descargados)}")

In [536]:
filename='BASE GENERAL.xlsx'
ruta_archivo = os.path.join(ruta_rrhh, filename)
df_embudo = pd.read_excel(ruta_archivo,sheet_name='EMBUDO JUNIO 2')

df_embudo.columns = [
    unidecode(col)          # quita tildes y ñ -> n
    .lower()                # minúsculas
    .strip()                # quita espacios al inicio y final
    .replace(" ", "_")      # espacios por _
    for col in df_embudo.columns
]

mapeo = {
    "telefono_1": "cel01",
    "telefono_2": "cel02",
}

df_embudo.rename(columns=mapeo, inplace=True)


columnas_fecha = [
    'fecha_de_inscripcion',
    'fecha_de_gestion',
    'fecha_entrevista',
    'fecha_inicio_de_capacitacion',
    'fecha_de_ingreso',
    'fecha_de_nacimiento'
]

for col in columnas_fecha:
    df_embudo[col] = pd.to_datetime(
        df_embudo[col],
        dayfirst=True,
        errors='coerce'
    )

df_embudo = df_embudo[
    df_embudo['reclutador'].notna() &
    df_embudo['fecha_de_gestion'].notna()
]

df_embudo['numero_de_documento'] = (
    df_embudo['numero_de_documento']
    .fillna('')
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
)

c:\Users\DATA\AppData\Local\Programs\Python\Python311\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [537]:

df_embudo['numero_de_documento'] = (
    df_embudo['numero_de_documento']
    .astype(str)
    .str.replace(r'\D', '', regex=True)      # deja solo números
    .replace('', pd.NA)                      # vacío -> NA
    .mask(lambda s: s.str.len() > 8, pd.NA)  # >8 dígitos -> NA
    .str.zfill(8)                            # <8 dígitos -> completa con ceros
)


In [538]:

for columna in ['cel01', 'cel02']:
    df_embudo[f'{columna}'] = (
        df_embudo[columna]
        .astype('string')
        .str.replace(r'\D', '', regex=True)
        .str.strip()
    )
df_embudo['cel01'] = df_embudo['cel01'].fillna(
    df_embudo['cel02']
)
df_embudo["edad"] = pd.to_numeric(
    df_embudo["edad"],
    errors="coerce"
)

In [539]:



bins = [17, 24, 29, 39, 49, 59, 65, np.inf]

labels = [
    "18 - 24",
    "25 - 29",
    "30 - 39",
    "40 - 49",
    "50 - 59",
    "60 - 65",
    "66+"
]

df_embudo["seg_edad"] = (
    pd.cut(
        df_embudo["edad"],
        bins=bins,
        labels=labels
    )
    .cat.add_categories("No registra")
    .fillna("No registra")
)

In [540]:
df_embudo["llave_postulante"] = (
    df_embudo["nombre"].fillna("").str.strip().str.upper() + "|" +
    df_embudo["apellido"].fillna("").str.strip().str.upper() + "|" +
    df_embudo["cel01"].fillna("").astype(str).str.strip()
)

dni_por_llave = (
    df_embudo
    .dropna(subset=["numero_de_documento"])
    .drop_duplicates("llave_postulante")
    .set_index("llave_postulante")["numero_de_documento"]
)

df_embudo["numero_de_documento"] = (
    df_embudo["numero_de_documento"]
    .fillna(df_embudo["llave_postulante"].map(dni_por_llave))
)

dni_invalidos = [
    "00000000",
    "99999999",
    "11111111",
    "12345678"
]

df_embudo["numero_de_documento"] = (
    df_embudo["numero_de_documento"]
    .replace(dni_invalidos, pd.NA)
)

df_embudo['id_postulante'] = np.where(
    df_embudo['numero_de_documento'].notna(),
    'DNI_' + df_embudo['numero_de_documento'],
    'TELF_' + df_embudo['cel01']
)
df_embudo["numero_de_documento"] = (
    df_embudo["numero_de_documento"]
    .fillna(df_embudo["cel01"])
)


## tb postulante

In [541]:
columnas_postulante = [
        'nombre',
        'apellido',
        'tipo_de_documento',
        'numero_de_documento',
        'fecha_de_inscripcion',
        'fuentes_laborales',
        'correo_electronico',
        'departamento',
        'distrito',
        'direccion',
        'fecha_de_nacimiento',
        'edad',
        'genero',
        'estado_civil',
        'nacionalidad',
        'fecha_de_gestion'
    ]

df_postulante = (
    df_embudo
    .sort_values(
        'fecha_de_gestion',
        ascending=False
    )[
        ['id_postulante'] + columnas_postulante
    ]
    .groupby(
        'id_postulante',
        as_index=False,
        sort=False
    )
    .first()
    .copy()
)
df_postulante = df_postulante.drop(columns='fecha_de_gestion')

In [542]:
columnas_eliminar = columnas_postulante.copy()
columnas_eliminar.remove('fuentes_laborales')
columnas_eliminar.remove('fecha_de_gestion')

df_embudo = (
    df_embudo
    .drop(columns=columnas_eliminar)
    .copy()
)

## tabla evento

In [543]:
df_embudo = df_embudo[
    (df_embudo['fecha_de_gestion'] >= pd.Timestamp('2026-06-30')) &
    (df_embudo['campana'].notna())&
    (df_embudo['reclutador'].notna())
]


In [544]:
columnas_evento = [
    'id_postulante',
    'campana',
    'fuentes_laborales',
    'modalidad',
    'jornada',
    'turno',
    'horario_de_trabajo',
    'planilla',
    'supervisor',
    'sede',
    'fecha_de_gestion',
    'reclutador',
    'fecha_entrevista',
    'entrevista',
    'status_entrevista',
    'fecha_inicio_de_capacitacion',
    'capacitacion',
    'motivo_capacitacion',
    'ojt',
    'motivo_ojt',
    'firma_de_contrato',
    'fecha_de_ingreso',
    'observacion'
]

df_evento = df_embudo[columnas_evento].copy()

fecha_gestion_ingreso = (
    df_evento.loc[df_evento['fecha_de_ingreso'].notna()]
    .groupby(['id_postulante', 'campana'])['fecha_de_gestion']
    .max()
    .rename('fecha_gestion_ingreso')
)

df_evento = df_evento.merge(
    fecha_gestion_ingreso,
    on=['id_postulante', 'campana'],
    how='left'
)


df_evento = df_evento[
    df_evento['fecha_gestion_ingreso'].isna()
    | (df_evento['fecha_de_gestion'] <= df_evento['fecha_gestion_ingreso'])
].copy()

df_evento = (
    df_evento
    .sort_values(
        ['id_postulante', 'campana', 'fecha_de_gestion'],
        ascending=[True, True, False]
    )
    .drop_duplicates(
        subset=['id_postulante', 'campana'],
        keep='first'
    )
    .drop(columns=['fecha_gestion_ingreso'])
    .rename(columns={'fecha_de_gestion': 'fecha_de_gestion_evento'})
    .reset_index(drop=True)
)

columnas_eliminar = columnas_evento.copy()
columnas_eliminar = [
    c for c in columnas_eliminar
    if c not in {
        'fecha_de_gestion',
        'fecha_de_ingreso',
        'id_postulante',
        'reclutador',
        'campana'
    }
]

df_embudo = (
    df_embudo
    .drop(columns=columnas_eliminar)
    .copy()
)

In [545]:
df_postulante = df_postulante[
    df_postulante['id_postulante'].isin(df_evento['id_postulante'])
].copy()

nro_campanas = (
    df_evento
    .groupby('id_postulante')['campana']
    .nunique()
)

df_postulante['nro_campanas'] = (
    df_postulante['id_postulante']
    .map(nro_campanas)
    .fillna(0)
    .astype('int16')
)

### tabla status_base

In [546]:
columnas_status = [
    'id_postulante',
    'campana',
    'reclutador',
    'llamada_1',
    'status_llamada_1',
    'llamada_2',
    'status_llamada_2',
    'mensaje_whats_app',
    'status_whats_app',
    'fecha_de_gestion',
    'cel01',
    'cel02'
]

df_status = df_embudo[columnas_status].copy()

columnas_status = [
    'status_llamada_1',
    'status_llamada_2',
    'status_whats_app'
]

for columna in columnas_status:
    df_status[columna] = (
        df_status[columna]
        .astype("string")
        .str.strip()
        .str.upper()
        .replace({
            "": pd.NA,
            "NAN": pd.NA,
            "NONE": pd.NA,
            "NULL": pd.NA
        })
    )
df_llamada_1 = (
    df_status[
        [
            'id_postulante',
            'campana',
            'reclutador',
            'fecha_de_gestion',
            'llamada_1',
            'status_llamada_1',
            'cel01',
            'cel02'
        ]
    ]
    .rename(columns={
        'reclutador':'reclutador_status',
        'llamada_1':'detalle_intento',
        'status_llamada_1':'status'
    })
)

df_llamada_1['tipo_intento'] = 'LLAMADA 1'
df_llamada_1['orden_intento'] = 1

df_llamada_2 = (
    df_status[
        [
            'id_postulante',
            'campana',
            'reclutador',
            'fecha_de_gestion',
            'llamada_2',
            'status_llamada_2',
            'cel01',
            'cel02'
        ]
    ]
    .rename(columns={
        'reclutador':'reclutador_status',
        'llamada_2':'detalle_intento',
        'status_llamada_2':'status'
    })
)

df_llamada_2['tipo_intento'] = 'LLAMADA 2'
df_llamada_2['orden_intento'] = 2

df_whatsapp = (
    df_status[
        [
            'id_postulante',
            'campana',
            'reclutador',
            'fecha_de_gestion',
            'mensaje_whats_app',
            'status_whats_app',
            'cel01',
            'cel02'
        ]
    ]
    .rename(columns={
        'reclutador':'reclutador_status',
        'mensaje_whats_app':'detalle_intento',
        'status_whats_app':'status'
    })
)

df_whatsapp['tipo_intento'] = 'WHATSAPP'
df_whatsapp['orden_intento'] = 3

df_status_consolidado = pd.concat(
    [
        df_llamada_1,
        df_llamada_2,
        df_whatsapp
    ],
    ignore_index=True
)
df_status_consolidado = (
    df_status_consolidado
    .dropna(subset=['status'])
    .reset_index(drop=True)
)


In [547]:
peso_status = {
    'CITADO': 1,
    'REPROGRAMAR LLAMADA': 2,
    'ESCRIBIR POR WHATS APP': 3,
    'LABORANDO ACTUALMENTE': 4,
    'DESEA REMOTO': 5,
    'DISTANCIA': 6,
    'NO INTERESADO EN SALARIO': 7,
    'NO INTERESADO EN PLANILLA MYPE': 8,
    'NO INTERESADO EN HORARIOS': 9,
    'NO INTERESADO EN VENTAS': 10,
    'NO INTERESADO EN LA PROPUESTA LABORAL': 11,
    'NO PERFIL': 12,
    'EDAD': 13,
    'GESTACIÓN': 14,
    'BLACK LIST': 15,
    'NO CONTACTADO': 99
}
df_status_consolidado['peso_status'] = (
    df_status_consolidado['status']
    .map(peso_status)
)

In [548]:
llaves = ['id_postulante', 'campana']

for intento, columna in {
    'LLAMADA 1': 'q_llamada_1',
    'LLAMADA 2': 'q_llamada_2',
    'WHATSAPP': 'q_whatsapp'
}.items():

    df_status_consolidado[columna] = (
        df_status_consolidado['tipo_intento']
        .eq(intento)
        .groupby([df_status_consolidado[c] for c in llaves])
        .transform('sum')
    )

In [549]:

df_status_mr = (
    df_status_consolidado
    .sort_values(
        llaves + [
            'peso_status',
            'fecha_de_gestion',
            'orden_intento'
        ],
        ascending=[
            True,
            True,
            True,
            False,
            False
        ],
        na_position='last'
    )
    .drop_duplicates(
        subset=llaves,
        keep='first'
    )
    [
        llaves + [
            'status',
            'reclutador_status',
            'peso_status',
            'fecha_de_gestion',
            'tipo_intento',
            'detalle_intento',
            'q_llamada_1',
            'q_llamada_2',
            'q_whatsapp'
        ]
    ]
    .rename(
        columns={
            'status': 'mejor_status',
            'peso_status': 'peso_mejor_status',
            'fecha_de_gestion': 'fecha_mejor_status',
            'tipo_intento': 'canal_mejor_status',
            'detalle_intento': 'detalle_mejor_intento'
        }
    )
)

In [550]:
df_evento = df_evento.merge(
    df_status_mr,
    on=['id_postulante', 'campana'],
    how='left'
)

In [551]:
df_evento['llave_postulante_campana'] = (
    df_evento['id_postulante']
    + '_'
    + df_evento['campana']
)
df_status_consolidado['llave_postulante_campana'] = (
    df_status_consolidado['id_postulante']
    + '_'
    + df_status_consolidado['campana']
)


In [552]:

df_postulante_campana = (
    df_evento[
        # ['llave_postulante_campana', 'id_postulante', 'campana']
        ['id_postulante', 'llave_postulante_campana', 'campana', 'jornada', 'turno', 'horario_de_trabajo', 'planilla', 'supervisor', 'sede', 'fecha_entrevista', 'capacitacion', 'motivo_capacitacion', 'ojt', 'motivo_ojt', 'observacion', 'reclutador', 'peso_mejor_status', 'fecha_mejor_status', 'modalidad','entrevista','reclutador_status','fuentes_laborales','fecha_de_ingreso','fecha_de_gestion_evento']

    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


In [553]:
print(df_status_consolidado.shape)
print(df_postulante.shape)
print(df_evento.shape)
print(df_postulante_campana.shape)


(1757, 15)
(1069, 17)
(1165, 33)
(1165, 24)


In [554]:
df_validar = df_postulante[
    ~df_postulante['id_postulante'].isin(df_evento['id_postulante'])
].copy()


In [555]:
df_validar.head()

,id_postulante,nombre,apellido,tipo_de_documento,numero_de_documento,fecha_de_inscripcion,fuentes_laborales,correo_electronico,departamento,distrito,direccion,fecha_de_nacimiento,edad,genero,estado_civil,nacionalidad,nro_campanas


In [556]:
print(df_evento.columns.tolist())
# print(df_status_mr.columns.tolist())
print(df_status_consolidado.columns.tolist())
print(df_postulante.columns.tolist())

['id_postulante', 'campana', 'fuentes_laborales', 'modalidad', 'jornada', 'turno', 'horario_de_trabajo', 'planilla', 'supervisor', 'sede', 'fecha_de_gestion_evento', 'reclutador', 'fecha_entrevista', 'entrevista', 'status_entrevista', 'fecha_inicio_de_capacitacion', 'capacitacion', 'motivo_capacitacion', 'ojt', 'motivo_ojt', 'firma_de_contrato', 'fecha_de_ingreso', 'observacion', 'mejor_status', 'reclutador_status', 'peso_mejor_status', 'fecha_mejor_status', 'canal_mejor_status', 'detalle_mejor_intento', 'q_llamada_1', 'q_llamada_2', 'q_whatsapp', 'llave_postulante_campana']
['id_postulante', 'campana', 'reclutador_status', 'fecha_de_gestion', 'detalle_intento', 'status', 'cel01', 'cel02', 'tipo_intento', 'orden_intento', 'peso_status', 'q_llamada_1', 'q_llamada_2', 'q_whatsapp', 'llave_postulante_campana']
['id_postulante', 'nombre', 'apellido', 'tipo_de_documento', 'numero_de_documento', 'fecha_de_inscripcion', 'fuentes_laborales', 'correo_electronico', 'departamento', 'distrito', 'd

In [557]:
from sqlalchemy import text

with engine_zeus.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS tb_rrhh_postulante"))
    conn.execute(text("DROP TABLE IF EXISTS tb_rrhh_proceso"))
    conn.execute(text("DROP TABLE IF EXISTS tb_rrhh_intentos"))
    conn.execute(text("DROP TABLE IF EXISTS tb_rrhh_postulante_campana"))
    # conn.execute(text("TRUNCATE TABLE tb_funnel_reclutamiento"))
    # conn.execute(text("TRUNCATE TABLE tb_funnel_reclutamiento"))
    # conn.execute(text("TRUNCATE TABLE tb_funnel_reclutamiento"))

In [558]:
display(df_postulante.head(3))
display(df_evento.head(3))
display(df_status_consolidado.head(3))
display(df_postulante_campana.head(3))

,id_postulante,nombre,apellido,tipo_de_documento,numero_de_documento,fecha_de_inscripcion,fuentes_laborales,correo_electronico,departamento,distrito,direccion,fecha_de_nacimiento,edad,genero,estado_civil,nacionalidad,nro_campanas
0,DNI_70329917,LUCIA YOLANDA,Zea Vilcas,DNI,70329917,2026-07-23,LINKED IN,luyozevi91@gmail.com,LIMA,VILLA EL SALVADOR,calle 74 mz b lt26 urb. pachacamac. villa el salvador,1991-03-14,35.0,FEMENINO,SOLTERO,PERUANA,1
1,DNI_73144879,Nadine,Huaman,DNI,73144879,2026-07-18,COMPUTRABAJO,nadinehuaman16@gmail.com,LIMA,COMAS,None,2006-03-16,20.0,FEMENINO,SOLTERO,PERUANA,2
2,DNI_05496458,Jahaziel,Aranguren,DNI,05496458,2026-07-23,COMPUTRABAJO,jahazielaranguren17@gmail.com,LIMA,COMAS,None,2007-07-18,19.0,MASCULINO,SOLTERO,PERUANA,1


,id_postulante,campana,fuentes_laborales,modalidad,jornada,turno,horario_de_trabajo,planilla,supervisor,sede,fecha_de_gestion_evento,reclutador,fecha_entrevista,entrevista,status_entrevista,fecha_inicio_de_capacitacion,capacitacion,motivo_capacitacion,ojt,motivo_ojt,firma_de_contrato,fecha_de_ingreso,observacion,mejor_status,reclutador_status,peso_mejor_status,fecha_mejor_status,canal_mejor_status,detalle_mejor_intento,q_llamada_1,q_llamada_2,q_whatsapp,llave_postulante_campana
0,DNI_01481748,EFECTIVA NEGOCIOS,COMPUTRABAJO,PRESENCIAL,SEMI FULL,TARDE,L - V 11:00 a 18:00; y S 9:00 a 14:01,MICRO,KRISSEL CÁRDENAS,LIMA,2026-07-17,KIMBHERLY CORRO,2026-07-18,PRESENTE,SE RETIRA DE LA ENTREVISTA,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,CITADO,KIMBHERLY CORRO,1.0,2026-07-17,LLAMADA 1,CONTESTA,1.0,0.0,0.0,DNI_01481748_EFECTIVA NEGOCIOS
1,DNI_02556064,DINERS SSFF,COMPUTRABAJO,PRESENCIAL,SEMI FULL,TARDE,11:00 AM A 06:00 PM,MICRO,JOHANA ARTEAGA,LIMA,2026-07-09,OCTAVIO RODRIGUEZ,NaT,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,DISTANCIA,OCTAVIO RODRIGUEZ,6.0,2026-07-09,LLAMADA 1,CONTESTA,1.0,0.0,0.0,DNI_02556064_DINERS SSFF
2,DNI_02556064,EFECTIVA NEGOCIOS,COMPUTRABAJO,PRESENCIAL,SEMI FULL,MAÑANA,11:00 am a 4:00 pm,MICRO,KRISSEL CÁRDENAS,LIMA,2026-07-15,KIMBHERLY CORRO,2026-06-16,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,CITADO,KIMBHERLY CORRO,1.0,2026-07-15,WHATSAPP,CONTESTA,2.0,2.0,2.0,DNI_02556064_EFECTIVA NEGOCIOS


,id_postulante,campana,reclutador_status,fecha_de_gestion,detalle_intento,status,cel01,cel02,tipo_intento,orden_intento,peso_status,q_llamada_1,q_llamada_2,q_whatsapp,llave_postulante_campana
0,DNI_71979884,DINERS TC,OCTAVIO RODRIGUEZ,2026-07-02,BLOQUEO DE LLAMDA POR SPAM,NO CONTACTADO,977668122,936724868,LLAMADA 1,1,99,1,1,0,DNI_71979884_DINERS TC
1,DNI_76861654,CREDICASH,OCTAVIO RODRIGUEZ,2026-07-01,CONTESTA,CITADO,991146765,<NA>,LLAMADA 1,1,1,1,0,0,DNI_76861654_CREDICASH
2,DNI_77158540,CENCOSUD SSFF,OCTAVIO RODRIGUEZ,2026-07-02,CONTESTA,CITADO,945184412,<NA>,LLAMADA 1,1,1,1,0,0,DNI_77158540_CENCOSUD SSFF


,id_postulante,llave_postulante_campana,campana,jornada,turno,horario_de_trabajo,planilla,supervisor,sede,fecha_entrevista,capacitacion,motivo_capacitacion,ojt,motivo_ojt,observacion,reclutador,peso_mejor_status,fecha_mejor_status,modalidad,entrevista,reclutador_status,fuentes_laborales,fecha_de_ingreso,fecha_de_gestion_evento
0,DNI_01481748,DNI_01481748_EFECTIVA NEGOCIOS,EFECTIVA NEGOCIOS,SEMI FULL,TARDE,L - V 11:00 a 18:00; y S 9:00 a 14:01,MICRO,KRISSEL CÁRDENAS,LIMA,2026-07-18,NaN,NaN,NaN,NaN,NaN,KIMBHERLY CORRO,1.0,2026-07-17,PRESENCIAL,PRESENTE,KIMBHERLY CORRO,COMPUTRABAJO,NaT,2026-07-17
1,DNI_02556064,DNI_02556064_DINERS SSFF,DINERS SSFF,SEMI FULL,TARDE,11:00 AM A 06:00 PM,MICRO,JOHANA ARTEAGA,LIMA,NaT,NaN,NaN,NaN,NaN,NaN,OCTAVIO RODRIGUEZ,6.0,2026-07-09,PRESENCIAL,NaN,OCTAVIO RODRIGUEZ,COMPUTRABAJO,NaT,2026-07-09
2,DNI_02556064,DNI_02556064_EFECTIVA NEGOCIOS,EFECTIVA NEGOCIOS,SEMI FULL,MAÑANA,11:00 am a 4:00 pm,MICRO,KRISSEL CÁRDENAS,LIMA,2026-06-16,NaN,NaN,NaN,NaN,NaN,KIMBHERLY CORRO,1.0,2026-07-15,PRESENCIAL,NaN,KIMBHERLY CORRO,COMPUTRABAJO,NaT,2026-07-15


In [559]:

df_postulante.to_sql(
    name="tb_rrhh_postulante",
    con=engine_zeus,
    if_exists="append",
    index=False,
    chunksize=1000
)

df_evento.to_sql(
    name="tb_rrhh_proceso",
    con=engine_zeus,
    if_exists="append",
    index=False,
    chunksize=1000
)
df_status_consolidado.to_sql(
    name="tb_rrhh_intentos",
    con=engine_zeus,
    if_exists="append",
    index=False,
    chunksize=1000
)
df_postulante_campana.to_sql(
    name="tb_rrhh_postulante_campana",
    con=engine_zeus,
    if_exists="append",
    index=False,
    chunksize=1000
)


121

In [ ]:
ruta_archivo = os.path.join(ruta_csv, 'negocios_1.xlsx')

df_postulante.to_excel(ruta_archivo, index=False)

In [ ]:
# display(df_evento.head(3))


,id_postulante,campana,modalidad,jornada,turno,horario_de_trabajo,planilla,supervisor,sede,fecha_de_gestion_evento,fecha_entrevista,entrevista,status_entrevista,fecha_inicio_de_capacitacion,capacitacion,motivo_capacitacion,ojt,motivo_ojt,firma_de_contrato,fecha_de_ingreso,observacion,mejor_status,reclutador,peso_mejor_status,fecha_mejor_status,canal_mejor_status,detalle_mejor_intento,q_llamada_1,q_llamada_2,q_whatsapp,llave_postulante_campana
0,DNI_01481748,EFECTIVA NEGOCIOS,PRESENCIAL,SEMI FULL,TARDE,L - V 11:00 a 18:00; y S 9:00 a 14:01,MICRO,KRISSEL CÁRDENAS,LIMA,2026-07-17,2026-07-18,PRESENTE,SE RETIRA DE LA ENTREVISTA,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,CITADO,KIMBHERLY CORRO,1.0,2026-07-17,LLAMADA 1,CONTESTA,1.0,0.0,0.0,DNI_01481748_EFECTIVA NEGOCIOS
1,DNI_02556064,DINERS SSFF,PRESENCIAL,SEMI FULL,TARDE,11:00 AM A 06:00 PM,MICRO,JOHANA ARTEAGA,LIMA,2026-07-09,NaT,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,DISTANCIA,OCTAVIO RODRIGUEZ,6.0,2026-07-09,LLAMADA 1,CONTESTA,1.0,0.0,0.0,DNI_02556064_DINERS SSFF
2,DNI_02556064,EFECTIVA NEGOCIOS,PRESENCIAL,SEMI FULL,MAÑANA,11:00 am a 4:00 pm,MICRO,KRISSEL CÁRDENAS,LIMA,2026-07-15,2026-06-16,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,CITADO,KIMBHERLY CORRO,1.0,2026-07-15,WHATSAPP,CONTESTA,2.0,2.0,2.0,DNI_02556064_EFECTIVA NEGOCIOS


In [521]:
df_evento[df_evento["id_postulante"]=='DNI_01481748'].head()

,id_postulante,campana,fuentes_laborales,modalidad,jornada,turno,horario_de_trabajo,planilla,supervisor,sede,fecha_de_gestion_evento,reclutador,fecha_entrevista,entrevista,status_entrevista,fecha_inicio_de_capacitacion,capacitacion,motivo_capacitacion,ojt,motivo_ojt,firma_de_contrato,fecha_de_ingreso,observacion,mejor_status,reclutador_status,peso_mejor_status,fecha_mejor_status,canal_mejor_status,detalle_mejor_intento,q_llamada_1,q_llamada_2,q_whatsapp,llave_postulante_campana
0,DNI_01481748,EFECTIVA NEGOCIOS,COMPUTRABAJO,PRESENCIAL,SEMI FULL,TARDE,L - V 11:00 a 18:00; y S 9:00 a 14:01,MICRO,KRISSEL CÁRDENAS,LIMA,2026-07-17,KIMBHERLY CORRO,2026-07-18,PRESENTE,SE RETIRA DE LA ENTREVISTA,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,CITADO,KIMBHERLY CORRO,1.0,2026-07-17,LLAMADA 1,CONTESTA,1.0,0.0,0.0,DNI_01481748_EFECTIVA NEGOCIOS


In [374]:
df_postulante_campana[df_postulante_campana['id_postulante']=='DNI_05966484'].head(3)

,id_postulante,llave_postulante_campana,campana,jornada,turno,horario_de_trabajo,planilla,supervisor,sede,fecha_entrevista,capacitacion,motivo_capacitacion,ojt,motivo_ojt,observacion,reclutador,peso_mejor_status,fecha_mejor_status,modalidad,entrevista
8,DNI_05966484,DNI_05966484_CENCOSUD SSFF,CENCOSUD SSFF,SEMI FULL,MAÑANA,11:00 AM A 06:00 PM,MICRO,JUAN CAMACHO,COMAS,NaT,NaN,NaN,NaN,NaN,APAGADO,FHERGIE INFANTE,99.0,2026-07-01,PRESENCIAL,NaN
9,DNI_05966484,DNI_05966484_DINERS TC,DINERS TC,FULL TIME,MAÑANA,9:00 am a 6:00 pm,RXH,JUAN CAMACHO,COMAS,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,PRESENCIAL,NaN
10,DNI_05966484,DNI_05966484_EFECTIVA NEGOCIOS,EFECTIVA NEGOCIOS,SEMI FULL,TARDE,11:00 AM A 06:00 PM,MICRO,KRISSEL CÁRDENAS,LIMA,NaT,NaN,NaN,NaN,NaN,NaN,FHERGIE INFANTE,11.0,2026-07-08,PRESENCIAL,NaN


In [375]:
df_evento[df_evento['id_postulante']=='DNI_05966484'].head(3)


,id_postulante,campana,modalidad,jornada,turno,horario_de_trabajo,planilla,supervisor,sede,fecha_de_gestion_evento,fecha_entrevista,entrevista,status_entrevista,fecha_inicio_de_capacitacion,capacitacion,motivo_capacitacion,ojt,motivo_ojt,firma_de_contrato,fecha_de_ingreso,observacion,mejor_status,reclutador,peso_mejor_status,fecha_mejor_status,canal_mejor_status,detalle_mejor_intento,q_llamada_1,q_llamada_2,q_whatsapp,llave_postulante_campana
8,DNI_05966484,CENCOSUD SSFF,PRESENCIAL,SEMI FULL,MAÑANA,11:00 AM A 06:00 PM,MICRO,JUAN CAMACHO,COMAS,2026-07-01,NaT,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT,APAGADO,NO CONTACTADO,FHERGIE INFANTE,99.0,2026-07-01,LLAMADA 1,APAGADO,1.0,0.0,0.0,DNI_05966484_CENCOSUD SSFF
9,DNI_05966484,DINERS TC,PRESENCIAL,FULL TIME,MAÑANA,9:00 am a 6:00 pm,RXH,JUAN CAMACHO,COMAS,2026-07-02,NaT,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,<NA>,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,DNI_05966484_DINERS TC
10,DNI_05966484,EFECTIVA NEGOCIOS,PRESENCIAL,SEMI FULL,TARDE,11:00 AM A 06:00 PM,MICRO,KRISSEL CÁRDENAS,LIMA,2026-07-08,NaT,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NO INTERESADO EN LA PROPUESTA LABORAL,FHERGIE INFANTE,11.0,2026-07-08,LLAMADA 1,CONTESTA,1.0,0.0,0.0,DNI_05966484_EFECTIVA NEGOCIOS


In [96]:
df_evento['entrevista'].unique()

array(['PRESENTE', nan, 'AUSENTE', 'REPROGRAMAR ENTREVISTA'], dtype=object)

In [98]:
resultado = (
    df_evento
    .groupby(['entrevista'])
    .size()
    .reset_index(name='cantidad')
)

print(resultado)

               entrevista  cantidad
0                 AUSENTE       105
1                PRESENTE       156
2  REPROGRAMAR ENTREVISTA         1


In [14]:
df_embudo=df_embudo.merge(df_llam_postulante_solo_dni,on=['vendor_lead_code','campana'],how='left')
df_embudo=df_embudo.merge(df_llam_postulante_todo,on=['vendor_lead_code_ref','campana'],how='left')


In [16]:
from sqlalchemy import text

with engine_zeus.begin() as conn:
    conn.execute(text("drop TABLE tb_funnel_reclutamiento"))
    # conn.execute(text("TRUNCATE TABLE tb_funnel_reclutamiento"))

In [17]:
df_embudo.columns = [
    unidecode(col)
    .lower()
    .strip()
    .replace(" ", "_")
    for col in df_embudo.columns
]

df_embudo["edad"] = pd.to_numeric(
    df_embudo["edad"],
    errors="coerce"
)

bins = [17, 24, 29, 39, 49, 59, 65, np.inf]

labels = [
    "18 - 24",
    "25 - 29",
    "30 - 39",
    "40 - 49",
    "50 - 59",
    "60 - 65",
    "66+"
]

df_embudo["seg_edad"] = (
    pd.cut(
        df_embudo["edad"],
        bins=bins,
        labels=labels
    )
    .cat.add_categories("No registra")
    .fillna("No registra")
)
df_embudo['tipo_base'] = np.where(
    df_embudo['vendor_lead_code'].isna(),
    'FUERA DE BASE',
    'EN BASE'
)

In [18]:
df_embudo["estado"] = df_embudo["estado"].fillna("NO GESTIONADO")
df_embudo["descripcion"] = df_embudo["descripcion"].fillna("PENDIENTE")
df_embudo["sub_estado"] = df_embudo["sub_estado"].fillna("PENDIENTE")
df_embudo["distrito"] = df_embudo["distrito"].fillna("Sin información")

In [19]:
dic_nombres = {
    "OCTAVIO RODRIGUEZ MORI": "OCTAVIO RODRIGUEZ",
    "KIMBHERLY GIOVANNA CORRO CARRASCO": "KIMBHERLY CORRO",
    "RAMON JOSE CRUZ QUISPE": "RAMÓN CRUZ"
}

mask = (
    df_embudo["reclutador"].isna() |
    (df_embudo["reclutador"].str.strip() == "")
)

df_embudo.loc[mask, "reclutador"] = (
    df_embudo.loc[mask, "ejecutivo"]
    .replace(dic_nombres)
)

In [20]:
import locale

locale.setlocale(locale.LC_TIME, "Spanish_Peru.1252")
df_embudo["fecha_de_gestion_bi"] = df_embudo["fecha_de_gestion"].dt.strftime("%Y-%m-%d")

df_embudo["dia"] = pd.to_datetime(df_embudo["fecha_de_gestion"]).dt.day
df_embudo["nombre_dia"] = pd.to_datetime(df_embudo["fecha_de_gestion"]).dt.day_name()

df_embudo["semana"] = pd.to_datetime(df_embudo["fecha_de_gestion"]).dt.isocalendar().week
df_embudo["semana_mes"] = (
    (pd.to_datetime(df_embudo["fecha_de_gestion"]).dt.day - 1) // 7 + 1
)

df_embudo["fecha_de_gestion"] = pd.to_datetime(df_embudo["fecha_de_gestion"])

meses = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril",
    5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto",
    9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"
}

df_embudo["mes"] = (
    df_embudo["fecha_de_gestion"].dt.strftime("%m") + " " +
    df_embudo["fecha_de_gestion"].dt.month.map(meses)
)

In [ ]:
# df_embudo["mes"] = pd.to_datetime(df_embudo["fecha_de_gestion"]).dt.month

In [35]:
df_embudo.head()

,reclutador,fecha_de_gestion,nombre,apellido,tipo_de_documento,numero_de_documento,fecha_de_inscripcion,postulaste_desde,correo_electronico,cel01,...,estado,descripcion,seg_edad,tipo_base,fecha_de_gestion_bi,dia,nombre_dia,semana,semana_mes,mes
0,KIMBHERLY CORRO,2026-07-02,Rebeca,Norifo,DNI,NaN,2026-07-02,NaN,NaN,912492759,...,NO GESTIONADO,PENDIENTE,No registra,EN BASE,2026-07-02,2.0,Thursday,27,1.0,7.0
1,OCTAVIO RODRIGUEZ,2026-07-06,Gabriela,Osorio,DNI,0,2026-07-06,COMPUTRABAJO,sharongaby.osorio@gmail.com,953084139,...,NO GESTIONADO,PENDIENTE,30 - 39,EN BASE,2026-07-06,6.0,Monday,28,1.0,7.0
2,OCTAVIO RODRIGUEZ,2026-06-24,Gabriela,Osorio,DNI,0,2026-05-27,BASE RECICLADA,sharongaby.osorio@gmail.com,953084139,...,NO GESTIONADO,PENDIENTE,30 - 39,EN BASE,2026-06-24,24.0,Wednesday,26,4.0,6.0
3,FHERGIE INFANTE,2026-07-06,Gabriela,Osorio,DNI,NaN,2026-07-06,COMPUTRABAJO,sharongaby.osorio@gmail.com,953084139,...,NO GESTIONADO,PENDIENTE,30 - 39,EN BASE,2026-07-06,6.0,Monday,28,1.0,7.0
4,FHERGIE INFANTE,2026-07-06,Karen,Franklink,DNI,1481748,2026-07-06,COMPUTRABAJO,kjfmmace@gmail.com,958279306,...,NO GESTIONADO,PENDIENTE,30 - 39,EN BASE,2026-07-06,6.0,Monday,28,1.0,7.0


In [72]:
df_embudo[df_embudo['numero_de_documento']=='76386419'].head()


,reclutador,fecha_de_gestion,nombre,apellido,tipo_de_documento,numero_de_documento,fecha_de_inscripcion,postulaste_desde,correo_electronico,cel01,...,estado,descripcion,seg_edad,tipo_base,fecha_de_gestion_bi,dia,nombre_dia,semana,semana_mes,mes
1573,OCTAVIO RODRIGUEZ,2026-07-02,René Roberto,Leyzaquia Ramos,DNI,76386419,2026-06-25,COMPUTRABAJO,reneleyzaquia18@gmail.com,913701896,...,NO GESTIONADO,PENDIENTE,18 - 24,EN BASE,2026-07-02,2.0,Thursday,27,1.0,7.0


In [ ]:

df_embudo.to_sql(
    name="tb_funnel_reclutamiento",
    con=engine_zeus,
    if_exists="append",
    index=False,
    chunksize=1000
)



41

In [35]:
df_embudo.shape

(1142, 65)

In [44]:
df_embudo[df_embudo['vendor_lead_code'].isnull()].head()

,reclutador,fecha_de_gestion,nombre,apellido,tipo_de_documento,vendor_lead_code,fecha_de_inscripcion,postulaste_desde,correo_electronico,cel01,...,ncant_todo,dni_ejecutivo,ejecutivo,duracion,fecha_llamada,phone_number,sub_estado,estado,descripcion,seg_edad
11,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,...,NaN,70564642,KIMBHERLY GIOVANNA CORRO CARRASCO,0.0,2026-06-11,42079074,No hubo contacto,NO CONTACTO,No contesta,No registra
20,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,...,NaN,77667905,RAMON JOSE CRUZ QUISPE,0.0,2026-06-01,51904280045,Objetivo final,CONTACTO EFECTIVO,Citado,No registra
21,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,...,NaN,77667905,RAMON JOSE CRUZ QUISPE,0.0,2026-06-02,51979482086,Existe interes,CONTACTO EFECTIVO,LLAMAR MÁS TARDE,No registra
66,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,...,NaN,70564642,KIMBHERLY GIOVANNA CORRO CARRASCO,0.0,2026-06-04,900216391,Dato incorrecto,CONTACTO NO EFECTIVO,Número no existe,No registra
67,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,...,NaN,70564642,KIMBHERLY GIOVANNA CORRO CARRASCO,16.0,2026-06-10,900387135,Dato incorrecto,CONTACTO NO EFECTIVO,Número equivocado,No registra


In [ ]:
# print(df_embudo["llamada_1"].unique().tolist())
# print(df_embudo["llamada_2"].unique().tolist())
print(df_embudo["capacitacion"].unique().tolist())

[nan, 'NO CALIFICA', 'NO ACEPTA PROPUESTA', 'DERIBAR A OTRA CUENTA', 'NO CONTACTADO', 'APTO', 'NO APTO', 'SE RETIRA DE LA ENTREVISTA', 'DESISTE']


In [ ]:
Contrato = 
CALCULATE(
    [OJT],
    tb_funnel_reclutamiento[capacitacion] IN {
        "SI"
    }
    
)

In [ ]:
postulaciones 
filtro cv  califica
contactados llamda12 whata contesto contesta
citados  status1 status2 status whatsaap citado
entrevistas presente
aptos  status_entrevista  apto
capacitacion EN PROCESO,COMPLETÓ,SE RETIRA DE LA CAPACITACIÓN
ASISTE PRIMER DÍA
ojt COMPLETÓ,NO COMPLETÓ,EN PROCESO
contratados firma_de_contrato SÍ




In [22]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

df_embudo.head()

,reclutador,fecha_de_gestion,nombre,apellido,tipo_de_documento,numero_de_documento,fecha_de_inscripcion,postulaste_desde,correo_electronico,cel01,cel02,departamento,distrito,direccion,fecha_de_nacimiento,edad,genero,estado_civil,nacionalidad,campana,modalidad,jornada,turno,horario_de_trabajo,planilla,supervisor,sede,filtro_cv,llamada_1,status_llamada_1,llamada_2,status_llamada_2,mensaje_whats_app,status_whats_app,fecha_entrevista,entrevista,status_entrevista,capacitacion,motivo_capacitacion,fecha_inicio_de_capacitacion,ojt,motivo_ojt,firma_de_contrato,fecha_de_ingreso,observacion,vendor_lead_code,vendor_lead_code_ref,fecha_de_inscripcion_final,ncant_solo_valido,ncant_todo,dni_ejecutivo,ejecutivo,duracion,fecha_llamada,phone_number,sub_estado,estado,descripcion,seg_edad,tipo_base
0,NaN,NaT,Vania,Allain Figueroa,DNI,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,CREDICASH,PRESENCIAL,PART TIME,TARDE,2:00 pm a 6:00 pm,RXH,JOSÉ LLANOS,COMAS,CALIFICA,CONTESTA,CITADO,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaT,NaN,00000000,00000000,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,PENDIENTE,NO GESTIONADO,PENDIENTE,No registra,EN BASE
1,KIMBHERLY CORRO,2026-06-11,Bibi,Khan,DNI,3524302,2026-06-11,COMPUTRABAJO,shaliza@outlook.es,929993643,NaN,LIMA,SAN JUAN DE LURIGANCHO,NaN,1988-03-31,38.0,FEMENINO,NaN,PERUANA,CENCOSUD SSFF,REMOTO,FULL TIME,MAÑANA,9:00 am a 6:00 pm,MICRO,ROSA NOVOA,LIMA,NO CALIFICA,CONTESTA,NO PERFIL,NO CALIFICA,NO CALIFICA,NO CALIFICA,NO CALIFICA,NaT,NO CALIFICA,NO CALIFICA,NO CALIFICA,NO CALIFICA,NaT,NO CALIFICA,NO CALIFICA,NO CALIFICA,NaT,EXTRANJERO,03524302,03524302,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,PENDIENTE,NO GESTIONADO,PENDIENTE,30 - 39,EN BASE
2,KIMBHERLY CORRO,2026-06-02,Josdaly Coromoto,Capella Figueredo,DNI,4285835,2026-05-27,COMPUTRABAJO,josscapella@gmail.com,916124014,NaN,LIMA,LIMA,NaN,1986-05-06,40.0,FEMENINO,NaN,PERUANA,DINERS SSFF,PRESENCIAL,FULL TIME,MAÑANA,9:00 am a 6:00 pm,RXH,JOHANA ARTEAGA,LIMA,CALIFICA,CONTESTA,NO INTERESADO EN VENTAS,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaT,NaN,04285835,04285835,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,PENDIENTE,NO GESTIONADO,PENDIENTE,40 - 49,EN BASE
3,KIMBHERLY CORRO,2026-06-01,Naudy Del Valle,Fernadez Armas,DNI,4499114,2026-05-28,COMPUTRABAJO,fernadezarmasnaudydelvalle@gmail.com,961162182,NaN,LIMA,SANTA ANITA,NaN,1994-09-10,31.0,FEMENINO,SOLTERO,PERUANA,DINERS SSFF,PRESENCIAL,FULL TIME,MAÑANA,9:00 am a 6:00 pm,RXH,JOHANA ARTEAGA,LIMA,NO CALIFICA,NO CALIFICA,NO CALIFICA,NO CALIFICA,NO CALIFICA,NO CALIFICA,NO CALIFICA,NaT,NO CALIFICA,NO CALIFICA,NO CALIFICA,NO CALIFICA,NaT,NaN,NaN,NaN,NaT,POSTULÓ A BACK OFFICE,04499114,04499114,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,PENDIENTE,NO GESTIONADO,PENDIENTE,30 - 39,EN BASE
4,KIMBHERLY CORRO,2026-06-06,Leticia Lizeth,Esquia Carasas,DNI,4712594,2026-06-05,COMPUTRABAJO,letizia2722@gmail.com,962590031,962590031,LIMA,LIMA,jr. general cordova 2233,1992-02-27,34.0,FEMENINO,SOLTERO,PERUANA,CENCOSUD SSFF,REMOTO,FULL TIME,MAÑANA,9:00 am a 6:00 pm,MICRO,ROSA NOVOA,LIMA,CALIFICA,CONTESTA,CITADO,NaN,NaN,NaN,NaN,2026-06-08,PRESENTE,NO ACEPTA PROPUESTA,NaN,NaN,NaT,NaN,NaN,NaN,NaT,NaN,04712594,04712594,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaN,PENDIENTE,NO GESTIONADO,PENDIENTE,30 - 39,EN BASE


In [66]:
print(df_embudo.columns.tolist())

['reclutador', 'fecha_de_gestion', 'nombre', 'apellido', 'tipo_de_documento', 'numero_de_documento', 'fecha_de_inscripcion', 'postulaste_desde', 'correo_electronico', 'cel01', 'cel02', 'departamento', 'distrito', 'direccion', 'fecha_de_nacimiento', 'edad', 'genero', 'estado_civil', 'nacionalidad', 'campana', 'modalidad', 'jornada', 'turno', 'horario_de_trabajo', 'planilla', 'supervisor', 'sede', 'filtro_cv', 'llamada_1', 'status_llamada_1', 'llamada_2', 'status_llamada_2', 'mensaje_whats_app', 'status_whats_app', 'fecha_entrevista', 'entrevista', 'status_entrevista', 'capacitacion', 'motivo_capacitacion', 'fecha_inicio_de_capacitacion', 'ojt', 'motivo_ojt', 'firma_de_contrato', 'fecha_de_ingreso', 'observacion', 'vendor_lead_code', 'vendor_lead_code_ref', 'fecha_de_inscripcion_final', 'ncant_solo_valido', 'ncant_todo', 'dni_ejecutivo', 'ejecutivo', 'duracion', 'fecha_llamada', 'phone_number', 'sub_estado', 'estado', 'descripcion', 'seg_edad', 'tipo_base', 'fecha_de_gestion_bi', 'dia', 

In [ ]:
['reclutador', 'fecha_de_gestion', 'nombre', 'apellido', 'tipo_de_documento', 'numero_de_documento', 'fecha_de_inscripcion', 'postulaste_desde', 'correo_electronico', 'cel01', 'cel02', 'departamento', 'distrito', 'direccion', 'fecha_de_nacimiento', 'edad', 'genero', 'estado_civil', 'nacionalidad', 'campana', 'modalidad', 'jornada', 'turno', 'horario_de_trabajo', 'planilla', 'supervisor', 'sede', 'filtro_cv', 'llamada_1', 'status_llamada_1', 'llamada_2', 'status_llamada_2', 'mensaje_whats_app', 'status_whats_app', 'fecha_entrevista', 'entrevista', 'status_entrevista', 'capacitacion', 'motivo_capacitacion', 'fecha_inicio_de_capacitacion', 'ojt', 'motivo_ojt', 'firma_de_contrato', 'fecha_de_ingreso', 'observacion', 'vendor_lead_code', 'vendor_lead_code_ref', 'fecha_de_inscripcion_final', 'ncant_solo_valido', 'ncant_todo', 'dni_ejecutivo', 'ejecutivo', 'duracion', 'fecha_llamada', 'phone_number', 'sub_estado', 'estado', 'descripcion', 'seg_edad', 'tipo_base', 'fecha_de_gestion_bi', 'dia', 'nombre_dia', 'semana', 'semana_mes']


In [140]:
print(df_embudo['status_llamada_1'].unique())

['NO CONTACTADO' 'NO CALIFICA' 'CITADO' 'NO PERFIL' 'DESEA REMOTO'
 'NO INTERESADO EN PLANILLA MYPE' 'PERFIL ESCUELITA'
 'NO INTERESADO EN SALARIO' nan 'NO INTERESADO EN HORARIOS'
 'LABORANDO ACTUALMENTE' 'DISTANCIA' 'NO INTERESADO EN VENTAS'
 'NO INTERESADO EN LA PROPUESTA LABORAL' 'BLACK LIST']


In [ ]:
df_embudo[
    df_embudo['vendor_lead_code'] == '77045774'
][[
'FECHA DE INSCRIPCIÓN',
'FECHA DE GESTIÓN',
 'FECHA ENTREVISTA',
 'FECHA INICIO DE CAPACITACION',
 'FECHA DE INGRESO',
 'RECLUTADOR',
 'POSTULASTE DESDE',
 'CAMPAÑA',
 'JORNADA',
 'MODALIDAD',
 'TURNO',
 'PLANILLA',
 'vendor_lead_code',
 'NOMBRE',
 'APELLIDO',
 'TIPO DE DOCUMENTO',
 'CORREO ELECTRÓNICO',
 'CEL01',
 'CEL02',
 'DEPARTAMENTO',
 'FECHA DE NACIMIENTO',
 'DISTRITO',
 'DIRECCIÓN',
 'EDAD',
 'GÉNERO',
 'ESTADO CIVIL',
 'NACIONALIDAD',
 'HORARIO DE TRABAJO',
 'SUPERVISOR',
 'vendor_lead_code_ref',
 'SEDE',
 'LLAMADA 1',
 'STATUS LLAMADA 1',
 'LLAMADA 2',
 'STATUS LLAMADA 2',
 'MENSAJE WHATS APP',
 'STATUS WHATS APP',
 'FILTRO CV',
 'ENTREVISTA',
 'STATUS ENTREVISTA',
 'CAPACITACIÓN',
 'MOTIVO CAPACITACIÓN',
 'OJT',
 'MOTIVO OJT',
 'FIRMA DE CONTRATO',
 'OBSERVACIÓN'
]].head()

,FECHA DE INSCRIPCIÓN,FECHA DE GESTIÓN,FECHA ENTREVISTA,FECHA INICIO DE CAPACITACION,FECHA DE INGRESO,RECLUTADOR,CAMPAÑA,vendor_lead_code,LLAMADA 1,STATUS LLAMADA 1,...,STATUS WHATS APP,FILTRO CV,ENTREVISTA,STATUS ENTREVISTA,CAPACITACIÓN,MOTIVO CAPACITACIÓN,OJT,MOTIVO OJT,FIRMA DE CONTRATO,OBSERVACIÓN
1,30/05/2026,01/06/2026,NaN,NaN,NaN,OCTAVIO RODRIGUEZ,CREDICASH,77045774,NO CONTESTA,NO CONTACTADO,...,NO CONTACTADO,CALIFICA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
123,03/06/2026,03/06/2026,NaN,NaN,NaN,KIMBHERLY CORRO,DINERS TC,77045774,NO CALIFICA,NO CALIFICA,...,NO CALIFICA,NO CALIFICA,NO CALIFICA,NO CALIFICA,NO CALIFICA,NO CALIFICA,NO CALIFICA,NO CALIFICA,NO CALIFICA,NaN
375,03/06/2026,09/06/2026,09/06/2026,NaN,NaN,RAMÓN CRUZ,EXPERTIS,77045774,CONTESTA,CITADO,...,CITADO,CALIFICA,AUSENTE,NO CALIFICA,NO CALIFICA,NO CALIFICA,NO CALIFICA,NO CALIFICA,NO CALIFICA,NO SE PRESENTO


KeyError: 'vendor_lead_code_ref'

In [ ]:
df_embudo['vendor_lead_code_ref'] = (
    df_embudo['vendor_lead_code']
    .mask(
        df_embudo['vendor_lead_code'].isna() | (df_embudo['vendor_lead_code'] == '0'),
        df_embudo['CEL01']
    )
)
df_embudo.head()


,RECLUTADOR,FECHA DE GESTIÓN,NOMBRE,APELLIDO,TIPO DE DOCUMENTO,vendor_lead_code,FECHA DE INSCRIPCIÓN,POSTULASTE DESDE,CORREO ELECTRÓNICO,CEL01,...,STATUS ENTREVISTA,CAPACITACIÓN,MOTIVO CAPACITACIÓN,FECHA INICIO DE CAPACITACION,OJT,MOTIVO OJT,FIRMA DE CONTRATO,FECHA DE INGRESO,OBSERVACIÓN,vendor_lead_code_ref
0,OCTAVIO RODRIGUEZ,01/06/2026,Katherine,Aguilar,DNI,70723259,01/06/2026,COMPUTRABAJO,hashy08082016@gmail.com,904571403,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"NO CUMPLE CON LA EDAD, SERIA SU PRIMER TRABAJO",70723259
1,OCTAVIO RODRIGUEZ,01/06/2026,Maria,Asencio,DNI,77045774,30/05/2026,COMPUTRABAJO,maria27.11asencio@gmail.com,997311939,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,77045774
2,KIMBHERLY CORRO,01/06/2026,Nardy,Curty Arteaga,DNI,74720395,01/06/2026,COMPUTRABAJO,nca-admi-098@hotmail.com,982037515,...,APTO,SE RETIRA DE LA CAPACITACIÓN,NO INDICA MOTIVO,04/06/2026,NO ASISTIÓ,DESISTE,NO,NaN,NaN,74720395
3,KIMBHERLY CORRO,01/06/2026,Brian,Hurtado Web,DNI,74095580,30/05/2026,COMPUTRABAJO,brianhurtadoweb@gmail.com,905970554,...,NO CALIFICA,NO CALIFICA,NO CALIFICA,NaN,NO CALIFICA,NO CALIFICA,NO CALIFICA,NaN,NO DESEA VENTAS,74095580
4,OCTAVIO RODRIGUEZ,01/06/2026,Jesús Brayan,Rojas cardenas,DNI,70986744,01/06/2026,COMPUTRABAJO,Rojasjesus2202@gmail.com,913841097,...,APTO,NO ASISTE,DESISTE,02/06/2026,NO ASISTIÓ,DESISTE,NO,NaN,NaN,70986744


In [53]:
df_embudo[
df_embudo.duplicated(subset='vendor_lead_code_ref', keep=False)
].sort_values('vendor_lead_code_ref').head()

,RECLUTADOR,FECHA DE GESTIÓN,NOMBRE,APELLIDO,TIPO DE DOCUMENTO,vendor_lead_code,FECHA DE INSCRIPCIÓN,POSTULASTE DESDE,CORREO ELECTRÓNICO,CEL01,...,STATUS ENTREVISTA,CAPACITACIÓN,MOTIVO CAPACITACIÓN,FECHA INICIO DE CAPACITACION,OJT,MOTIVO OJT,FIRMA DE CONTRATO,FECHA DE INGRESO,OBSERVACIÓN,vendor_lead_code_ref
510,KIMBHERLY CORRO,12/06/2026,Paulo Enrique,Villegas Euribe,DNI,10784563,12/06/2026,COMPUTRABAJO,paulo.villegas1@gmail.com,980462909,...,NO APTO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CONFLICTO DE HORARIOS CON EMPLEO ACTUAL,10784563
489,KIMBHERLY CORRO,11/06/2026,Paulo,Villegas,DNI,10784563,11/06/2026,COMPUTRABAJO,paulo.villegas1@gmail.com,980462909,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SOLO MODALIDAD REMOTA,10784563
117,KIMBHERLY CORRO,02/06/2026,Danilo Mirko,Soto Bellido,DNI,41420423,27/05/2026,COMPUTRABAJO,danirko1981@gmail.com,930993992,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO TIENE EXPERIENCIA EN VENTAS / DESEA BACK OF...,41420423
77,KIMBHERLY CORRO,02/06/2026,Danilo Mirko,Soto Bellido,DNI,41420423,27/05/2026,COMPUTRABAJO,danirko1981@gmail.com,930993992,...,NO CALIFICA,NO CALIFICA,NO CALIFICA,NaN,NO CALIFICA,NO CALIFICA,NO CALIFICA,NaN,NO LE GUSTAN LAS VENTAS,41420423
89,KIMBHERLY CORRO,02/06/2026,Josdaly Coromoto,Capella Figueredo,DNI,4285835,27/05/2026,COMPUTRABAJO,josscapella@gmail.com,916124014,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4285835


In [38]:
df_vicidial[df_vicidial['vendor_lead_code'].isnull()]['phone_number'].nunique()

142

In [ ]:
df_vicidial[df_vicidial['vendor_lead_code'].isnull()].count()


dial_method           336
numero_campana        336
dni_ejecutivo         336
ejecutivo             336
nombre_campana        336
fecha_hora_llamada    336
duracion              335
fecha_llamada         336
call_result             2
list_description        2
list_name             336
phone_number          336
codigo                336
estado                336
descripcion           336
vendor_lead_code        0
dtype: int64

In [ ]:
['dial_method', 'numero_campana', 'dni_ejecutivo', 'ejecutivo', 'nombre_campana', 'fecha_hora_llamada', 'duracion', 'fecha_llamada', 'call_result', 'list_description', 'list_name', 'phone_number', 'codigo', 'estado', 'descripcion']

['dial_method', 'numero_campana', 'dni_ejecutivo', 'ejecutivo', 'nombre_campana', 'fecha_hora_llamada', 'duracion', 'fecha_llamada', 'call_result', 'list_description', 'list_name', 'phone_number', 'codigo', 'estado', 'descripcion']


In [ ]:
df_tnumero=df_embudo[['vendor_lead_code','CEL01','CEL02']]

In [ ]:
['RECLUTADOR', 'FECHA DE GESTIÓN', 'NOMBRE', 'APELLIDO', 'TIPO DE DOCUMENTO', 'NUMERO DE DOCUMENTO', 'FECHA DE INSCRIPCIÓN', 'POSTULASTE DESDE', 'CORREO ELECTRÓNICO', 'TELÉFONO 1', 'TELÉFONO 2', 'DEPARTAMENTO', 'DISTRITO', 'DIRECCIÓN', 'FECHA DE NACIMIENTO', 'EDAD', 'GÉNERO', 'ESTADO CIVIL', 'NACIONALIDAD', 'CAMPAÑA', 'MODALIDAD', 'JORNADA', 'TURNO', 'HORARIO DE TRABAJO', 'PLANILLA', 'SUPERVISOR', 'SEDE', 'FILTRO CV', 'LLAMADA 1', 'STATUS LLAMADA 1', 'LLAMADA 2', 'STATUS LLAMADA 2', 'MENSAJE WHATS APP', 'STATUS WHATS APP', 'FECHA ENTREVISTA', 'ENTREVISTA', 'STATUS ENTREVISTA', 'CAPACITACIÓN', 'MOTIVO CAPACITACIÓN', 'FECHA INICIO DE CAPACITACION', 'OJT', 'MOTIVO OJT', 'FIRMA DE CONTRATO', 'FECHA DE INGRESO', 'OBSERVACIÓN']


In [23]:
df_vicidial[df_vicidial['descripcion'].isnull()].head()


,vendor_lead_code,dial_method,numero_campana,dni_ejecutivo,ejecutivo,nombre_campana,fecha_hora_llamada,duracion,fecha_llamada,call_result,list_description,list_name,phone_number,codigo,estado,descripcion


In [ ]:

# print(f"df_dni filas: {df_long.shape[0]}")
# print(f"df_list filas: {df_list.shape[0]}")


In [146]:
print(sorted(df_embudo["observacion"].dropna().unique()))

['2 MESES DE EXPERIENCIA EN CALL', 'ALTA ROTACIÓN LABORAL', 'ATC EN EL SECTOR EDUCATIVO', 'BLACK LIST', 'BLACKLIST', 'BUSCO ALGO MUCHO MEJOR', 'CAPACITACIONES POR LA TARDE', 'CLARIDAD VERBAL NO ALINEADA AL PUESTO', 'COBRANZAS', 'COMUNICACIÓN VERBAL CON BAJA PROYECCIÓN', 'CON EXPERIENCIA EN ATC / VIVE EN PROVINCIA', 'CONFLICTO DE HORARIOS CON EMPLEO ACTUAL', 'CONSIDERAR PARA UN PRÓXIMO INICIO', 'CORTA EXPERIENCIA EN CALL CENTER', 'CORTA EXPERIENCIA EN CALL CENTER / NO RUBRO FINANCIERO', 'CORTA EXPERIENCIA EN FINANCIERA', 'CORTA EXPERIENCIA EN FINANZAS (DERIVACIÓN)', 'DERIVACIÓN DE DINERS A CREDICASH / SONDEAR EXPERIENCIA', 'DERIVAR A ALFIN BANCO', 'DERIVAR A CREDICASH O EFECTIVA', 'DESEA PART TIME REMOTO', 'DESISTE POR EL PAGO DE 5 Y 20 ', 'DESISTE PORQUE QUIERE PLANILLA COMPLETA', 'DESISTIO POR PLANILLA ', 'DINERS SSFF A CREDICASH', 'DISTANCIA', 'DISTANCIA / ACTUALMENTE TRABAJANDO', 'EN SU CV SE ESPECIFICA QUE PUEDE DESPUES DE LAS 5 LOS MARTES, JUEVES ', 'ENTREVISTA LUNES 09', 'ESTUDIA

['RECLUTADOR', 'FECHA DE GESTIÓN', 'NOMBRE', 'APELLIDO', 'TIPO DE DOCUMENTO', 'NUMERO DE DOCUMENTO', 'FECHA DE INSCRIPCIÓN', 'POSTULASTE DESDE', 'CORREO ELECTRÓNICO', 'TELÉFONO 1', 'TELÉFONO 2', 'DEPARTAMENTO', 'DISTRITO', 'DIRECCIÓN', 'FECHA DE NACIMIENTO', 'EDAD', 'GÉNERO', 'ESTADO CIVIL', 'NACIONALIDAD', 'CAMPAÑA', 'MODALIDAD', 'JORNADA', 'TURNO', 'HORARIO DE TRABAJO', 'PLANILLA', 'SUPERVISOR', 'SEDE', 'FILTRO CV', 'LLAMADA 1', 'STATUS LLAMADA 1', 'LLAMADA 2', 'STATUS LLAMADA 2', 'MENSAJE WHATS APP', 'STATUS WHATS APP', 'FECHA ENTREVISTA', 'ENTREVISTA', 'STATUS ENTREVISTA', 'CAPACITACIÓN', 'MOTIVO CAPACITACIÓN', 'FECHA INICIO DE CAPACITACION', 'OJT', 'MOTIVO OJT', 'FIRMA DE CONTRATO', 'FECHA DE INGRESO', 'OBSERVACIÓN']


In [ ]:


filename='EMBUDO.xlsx'
name_dni='DNI'

ruta_archivo = os.path.join(ruta_csv, filename)
df = pd.read_excel(ruta_archivo)

df[f"{name_dni}"] = (
    df[f"{name_dni}"]
    .astype(str)
    .str.zfill(8)
)

df = df.rename(columns={
    f'{name_dni}': 'NUMERO_DOCUMENTO'
})
df=df[["NUMERO_DOCUMENTO"]]
df.head()
server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

tipi_cond1='RECLUTA'
servidor_01=64
query = f"""
    SELECT *
    FROM OPENQUERY([192.168.3.{servidor_01}], '
        SELECT        
        rtrim(ltrim(d.vendor_lead_code)) AS vendor_lead_code,        
        e.dial_method,
        a.campaign_id AS numero_campana,        
        a.user AS dni_ejecutivo,
        c.full_name AS ejecutivo,
        e.campaign_name AS nombre_campana,        
        a.call_date AS fecha_hora_llamada,        
        a.length_in_sec AS duracion,        
        b.status_name AS call_result,        
        f.list_description,        
        f.list_name,        
        a.phone_number as phone_number,        
        d.alt_phone as fecha_agenda,        
        d.comments as comentarios,        
        a.status AS codigo
        FROM asterisk.vicidial_log a         
        LEFT JOIN asterisk.vicidial_list d ON a.lead_id=d.lead_id        
        LEFT JOIN asterisk.vicidial_campaigns e ON a.campaign_id=e.campaign_id        
        LEFT JOIN asterisk.vicidial_lists f ON a.list_id=f.list_id        
        LEFT JOIN asterisk.vicidial_statuses b ON a.status=b.status        
        LEFT JOIN asterisk.vicidial_users c ON a.user=c.user        
        WHERE e.campaign_name like "%{tipi_cond1}" 
        AND a.call_date >= DATE_FORMAT(''{fecha_mes_base}'', ''%Y-%m-01'')
        AND a.call_date < 
        DATE_ADD(DATE_FORMAT(''{fecha_mes_base}'', ''%Y-%m-01''), INTERVAL 1 MONTH)
    ')

    """
df_efe = pd.read_sql(query, engine_kishin)

df = df.merge(df_efe, on='NUMERO_DOCUMENTO', how='inner')
list_dni = (
    df['NUMERO_DOCUMENTO']
    .dropna()
    .drop_duplicates()
    .tolist()
)
# in_clause = ",".join(f"'{x}'" for x in list_dni)
valores = ",\n".join(f"('{dni}')" for dni in list_dni)


In [4]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

In [5]:
fecha_mes_base='2026-06-01'
tipi_cond1='RECLUTAMIENTO'
tipi_cond2='xx'
tipi_cond3='xx'
tb_tipolofia='tTipologia_Reclutamiento'
servidor_01=64
tipi_cod='cod'
tipi_resp_cod='RC0'
tipi_descrip='[NIVEL 4]'
tipi_estado='[NIVEL 2]'
tipi_resp_estado='NO CONTACTO'
tipi_subdescripcion='[NIVEL 3]'
tnum_tb='tNumeroDinersTc'
tnum_dni='NUMERO_DOCUMENTO'
tlista_generada='borrar_tc_dinner'
get_base=since_base_maestra_tc_dinners

query = f"""
    SELECT *
    FROM OPENQUERY([192.168.3.{servidor_01}], '
        SELECT        
        rtrim(ltrim(d.vendor_lead_code)) AS vendor_lead_code,        
        e.dial_method,
        a.campaign_id AS numero_campana,        
        a.user AS dni_ejecutivo,
        c.full_name AS ejecutivo,
        e.campaign_name AS nombre_campana,        
        a.call_date AS fecha_hora_llamada,        
        a.length_in_sec AS duracion,        
        b.status_name AS call_result,        
        f.list_description,        
        f.list_name,        
        a.phone_number as phone_number,        
        d.alt_phone as fecha_agenda,        
        d.comments as comentarios,        
        a.status AS codigo
        FROM asterisk.vicidial_log a         
        LEFT JOIN asterisk.vicidial_list d ON a.lead_id=d.lead_id        
        LEFT JOIN asterisk.vicidial_campaigns e ON a.campaign_id=e.campaign_id        
        LEFT JOIN asterisk.vicidial_lists f ON a.list_id=f.list_id        
        LEFT JOIN asterisk.vicidial_statuses b ON a.status=b.status        
        LEFT JOIN asterisk.vicidial_users c ON a.user=c.user        
        WHERE (e.campaign_name like "%{tipi_cond1}" or e.campaign_name like "%{tipi_cond2}" or e.campaign_name like "%{tipi_cond3}")
        AND a.call_date >= DATE_FORMAT(''{fecha_mes_base}'', ''%Y-%m-01'')
        AND a.call_date < 
        DATE_ADD(DATE_FORMAT(''{fecha_mes_base}'', ''%Y-%m-01''), INTERVAL 1 MONTH)
    ')

    """
df_vicidial=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

In [7]:
df_vicidial.show()

+----------------+--------------------+--------------+-------------+--------------------+--------------+-------------------+--------+-----------+----------------+-------------------+------------+------------+-----------+------+
|vendor_lead_code|         dial_method|numero_campana|dni_ejecutivo|           ejecutivo|nombre_campana| fecha_hora_llamada|duracion|call_result|list_description|          list_name|phone_number|fecha_agenda|comentarios|codigo|
+----------------+--------------------+--------------+-------------+--------------------+--------------+-------------------+--------+-----------+----------------+-------------------+------------+------------+-----------+------+
|                |MANUAL           ...|            71|     70198430|OCTAVIO RODRIGUEZ...| RECLUTAMIENTO|2026-06-01 09:48:02|       0|       NULL|            NULL|Default Manual list|   978379897|            |           | RC003|
|                |MANUAL           ...|            71|     70198430|OCTAVIO RODRIGUEZ...

In [3]:


# def resumen_vicidial(spark,fecha_mes_base,tipi_cond1,tipi_cond2,tipi_cond3,tb_tipolofia,servidor_01,tipi_cod,tipi_resp_cod,tipi_descrip,tipi_estado,tipi_resp_estado):
query = f"""
    SELECT *
    FROM OPENQUERY([192.168.3.{servidor_01}], '
        SELECT        
        rtrim(ltrim(d.vendor_lead_code)) AS vendor_lead_code,        
        e.dial_method,
        a.campaign_id AS numero_campana,        
        a.user AS dni_ejecutivo,
        c.full_name AS ejecutivo,
        e.campaign_name AS nombre_campana,        
        a.call_date AS fecha_hora_llamada,        
        a.length_in_sec AS duracion,        
        b.status_name AS call_result,        
        f.list_description,        
        f.list_name,        
        a.phone_number as phone_number,        
        d.alt_phone as fecha_agenda,        
        d.comments as comentarios,        
        a.status AS codigo
        FROM asterisk.vicidial_log a         
        LEFT JOIN asterisk.vicidial_list d ON a.lead_id=d.lead_id        
        LEFT JOIN asterisk.vicidial_campaigns e ON a.campaign_id=e.campaign_id        
        LEFT JOIN asterisk.vicidial_lists f ON a.list_id=f.list_id        
        LEFT JOIN asterisk.vicidial_statuses b ON a.status=b.status        
        LEFT JOIN asterisk.vicidial_users c ON a.user=c.user        
        WHERE (e.campaign_name like "%{tipi_cond1}" or e.campaign_name like "%{tipi_cond2}" or e.campaign_name like "%{tipi_cond3}")
        AND a.call_date >= DATE_FORMAT(''{fecha_mes_base}'', ''%Y-%m-01'')
        AND a.call_date < 
        DATE_ADD(DATE_FORMAT(''{fecha_mes_base}'', ''%Y-%m-01''), INTERVAL 1 MONTH)
    ')

    """
df_vicidial=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

    df_vicidial = df_vicidial.withColumn(
        "vendor_lead_code",
        F.lpad(F.col("vendor_lead_code").cast("string"), 8, "0")
    )
    query = f"""
        SELECT {tipi_cod} as codigo
        , case
            when {tipi_cod}='CALLBK' then 'VOLVER A LLAMAR - call'
            else {tipi_descrip} 
        end as descripcion
        ,case 
            when {tipi_cod}='CALLBK' then 1200
            else peso 
        end as peso  FROM [ODIN].[dbo].{tb_tipolofia}
        where LEFT({tipi_cod},2)='{tipi_resp_cod}' or {tipi_estado}='{tipi_resp_estado}' or {tipi_cod}='CALLBK'
        """
    df_tipi=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

    df_vicidial=df_vicidial.join(df_tipi,["codigo"],"left")

    return df_vicidial.select('fecha_hora_llamada','list_name','vendor_lead_code','phone_number','descripcion','dni_ejecutivo','ejecutivo','dial_method','call_result','duracion','codigo','nombre_campana')


IndentationError: unexpected indent (3305912164.py, line 36)

In [1]:
import os
import requests
import pandas as pd
import msal
from dotenv import load_dotenv

load_dotenv()

TENANT_ID = os.getenv("TENANT_ID")
CLIENT_ID = os.getenv("CLIENT_ID")

AUTHORITY = f"https://login.microsoftonline.com/{TENANT_ID}"
SCOPES = ["Files.Read.All", "Sites.Read.All"]

app = msal.PublicClientApplication(
    client_id=CLIENT_ID,
    authority=AUTHORITY
)

result = app.acquire_token_interactive(scopes=SCOPES)

if "access_token" not in result:
    raise Exception(result)

token = result["access_token"]

headers = {
    "Authorization": f"Bearer {token}"
}

ValueError: Unable to get authority configuration for https://login.microsoftonline.com/None. Authority would typically be in a format of https://login.microsoftonline.com/your_tenant or https://tenant_name.ciamlogin.com or https://tenant_name.b2clogin.com/tenant.onmicrosoft.com/policy.  Also please double check your tenant name or GUID is correct.

In [ ]:
url = "https://graph.microsoft.com/v1.0/me/drive/root:/Forms/NOMBRE_ARCHIVO.xlsx:/content"

response = requests.get(url, headers=headers)

with open("respuestas_forms.xlsx", "wb") as f:
    f.write(response.content)

df = pd.read_excel("respuestas_forms.xlsx")

print(df.head())

In [ ]:
pip install msal requests pandas openpyxl python-dotenv

In [ ]:
import base64
import requests
import pandas as pd
import msal

TENANT_ID = "TU_TENANT_ID"
CLIENT_ID = "TU_CLIENT_ID"

excel_url = """https://netorgft16620308-my.sharepoint.com/:x:/r/personal/seleccion_targetoutsourcing_org/_layouts/15/Doc.aspx?sourcedoc=%7BB42A55BF-161A-42B9-8592-5322FB3A3F09%7D&file=ENCUESTA%20DE%20SATISFACCI%C3%93N%20-%20TARGET%20OUTSOURCING.xlsx&action=edit&mobileredirect=true"""

authority = f"https://login.microsoftonline.com/{TENANT_ID}"
scopes = ["Files.Read.All", "Sites.Read.All"]

app = msal.PublicClientApplication(
    client_id=CLIENT_ID,
    authority=authority
)

result = app.acquire_token_interactive(scopes=scopes)

if "access_token" not in result:
    raise Exception(result)

token = result["access_token"]

headers = {
    "Authorization": f"Bearer {token}"
}

# Convertir URL a shareId
encoded_url = base64.urlsafe_b64encode(excel_url.encode()).decode().rstrip("=")
share_id = f"u!{encoded_url}"

# Descargar Excel
download_url = f"https://graph.microsoft.com/v1.0/shares/{share_id}/driveItem/content"

response = requests.get(download_url, headers=headers)

if response.status_code != 200:
    print(response.status_code)
    print(response.text)
    raise Exception("No se pudo descargar el archivo")

archivo_local = "respuestas_forms.xlsx"

with open(archivo_local, "wb") as f:
    f.write(response.content)

df = pd.read_excel(archivo_local)

print(df.head())

## Inicio

pip install Office365-REST-Python-Client pandas openpyxl

In [ ]:
import pandas as pd
from office365.sharepoint.client_context import ClientContext
from office365.runtime.auth.user_credential import UserCredential

usuario = "seleccion@targetoutsourcing.org"
password = "Target.2025$"

site_url = "https://netorgft16620308-my.sharepoint.com/personal/seleccion_targetoutsourcing_org"

file_url = "/personal/seleccion_targetoutsourcing_org/Documents/ENCUESTA DE SATISFACCIÓN - TARGET OUTSOURCING.xlsx"

# Archivo local donde se descargará
archivo_local = "respuestas_forms.xlsx"

ctx = ClientContext(site_url).with_credentials(
    UserCredential(usuario, password)
)

with open(archivo_local, "wb") as local_file:
    file = ctx.web.get_file_by_server_relative_url(file_url)
    file.download(local_file).execute_query()

print("Archivo descargado correctamente")

df = pd.read_excel(archivo_local)

print(df.head())

An error occurred while retrieving token from XML response: AADSTS90023: Invalid STS request.


ValueError: An error occurred while retrieving token from XML response: AADSTS90023: Invalid STS request.

### De nuevo